# Amir's modelling experiment: combine and filter 183 features

This notebook follows `templates/modelling.ipynb` and uses the shared
`BattleExperiment` runner. It combines Amir's 45 features, Aimilia's 106, and Leonard's 32 and compares
Logistic Regression with Histogram Gradient Boosting.

**Workflow:** extract 183 features → remove redundant features within each training fold → compare both candidates using five-fold
cross-validation (CV) on development data → explicitly evaluate the chosen model
on the holdout → optionally refit on all labelled battles and create a submission.

Holdout evaluation and submission generation are **off by default**. All artifacts
are saved under a unique `outputs/amirhosein/<run>/` folder in this checkout.
Previous scores from the standalone notebook do not apply to this workflow.

The feature covariance/correlation section uses only development rows; holdout
labels and test battles are excluded.


In [ ]:
from pathlib import Path
from datetime import datetime
from collections import Counter
import hashlib
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").is_file()
             and (p / "src/pokemon_challenge").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this repository folder or run from a notebook inside it.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from pokemon_challenge.experiment import BattleExperiment, Candidate
from pokemon_challenge.paths import data_dir
from pokemon_challenge.preparation.pokemon_details import default_output_dir, load_pokemon_details

OWNER = "amirhosein"
RUN = datetime.now().strftime("%Y%m%d_%H%M%S_%f_filtered_features")
DATA_DIR = data_dir()  # data/raw/, data/, Kaggle, or POKEMON_DATA_DIR
print("Project:", ROOT)
print("Data:", DATA_DIR)
print("Run:", OWNER, RUN)


## Context & Methods

Amir's feature block has **45 numeric inputs**: the original 43 inputs plus one type-effectiveness
total for each player. `notype` is a missing-secondary-type
placeholder and is excluded. The requested `p1-switch` spelling is normalized to
`p1_switch`. Species names are used only to look up defender types, not as model inputs.
No outcome-derived rates enter the models.

| Features | Definition |
| --- | --- |
| `p1_mean_{hp,atk,def,spa,spd,spe}` | Mean base stat over P1's roster |
| `p1_type_{type}_count` | Number of P1 roster members carrying each real type |
| `p1_team_diversity` | Number of distinct real types in P1's roster |
| `p2_lead_{hp,atk,def,spa,spd,spe}` | Supplied P2 lead's base stats |
| `p1_alive_at_turn30`, `p2_alive_at_turn30` | Six minus distinct Pokémon observed fainted |
| `alive_diff_at_turn30` | P1 remaining count minus P2 remaining count |
| `p1_mean_boosts`, `p2_mean_boosts` | Mean across turns of the sum of the five boost stages |
| `p1_no_of_moves`, `p2_no_of_moves` | Number of non-null recorded moves |
| `p1_dmg`, `p2_dmg` | Positive observed opponent HP drops, summed by Pokémon |
| `p1_heal`, `p2_heal` | Positive observed own HP increases, summed by Pokémon |
| `p1_switch`, `p2_switch` | Consecutive active-name changes, including forced replacements |
| `p1_type_effectiveness`, `p2_type_effectiveness` | Sum of move-versus-defender type scores over non-null moves; the Curse type contributes zero |

### Key assumptions

All timeline features stop after the first 30 entries. Fainted means zero HP or
status `fnt`, with each name counted once. Sleep and freeze do not imply fainting;
unseen members are presumed remaining in a six-member team.

Damage/healing compare the same Pokémon's successive observations, including
returns after switching. A newly seen Pokémon starts from an assumed HP of 1.0,
matching the earlier project's depletion proxy. These totals include changes
from unobserved intervals and do not prove which move caused them. HP units are
fractions of a Pokémon's maximum, not absolute HP points.

### Shared feature interface

`extract_selected_features` takes one battle and returns the same 45 numeric keys
for every observation. The shared runner removes the ID and target before calling
it and prefixes its output columns with `selected__`. No learned statistics or
outcome-derived rates are computed here.


In [ ]:
TEAM_SIZE = 6
MAX_OBSERVED_TURNS = 30

STATS = ('hp', 'atk', 'def', 'spa', 'spd', 'spe')
TYPES = (
    'bug', 'dark', 'dragon', 'electric', 'fighting', 'fire', 'flying',
    'ghost', 'grass', 'ground', 'ice', 'normal', 'poison', 'psychic',
    'rock', 'steel', 'water',
)
BOOST_STATS = ('atk', 'def', 'spa', 'spd', 'spe')
FEATURE_COLUMNS = (
    [f'p1_mean_{stat}' for stat in STATS]
    + [f'p1_type_{kind}_count' for kind in TYPES]
    + ['p1_team_diversity']
    + [f'p2_lead_{stat}' for stat in STATS]
    + ['p1_alive_at_turn30', 'p2_alive_at_turn30', 'alive_diff_at_turn30']
    + ['p1_mean_boosts', 'p2_mean_boosts']
    + ['p1_no_of_moves', 'p2_no_of_moves']
    + ['p1_dmg', 'p1_heal', 'p2_dmg', 'p2_heal', 'p1_switch', 'p2_switch']
    + ['p1_type_effectiveness', 'p2_type_effectiveness']
)
assert len(FEATURE_COLUMNS) == len(set(FEATURE_COLUMNS)) == 45


def extract_selected_features(battle):
    roster = battle.get('p1_team_details') or []
    lead = battle.get('p2_lead_details') or {}
    timeline = (battle.get('battle_timeline') or [])[:MAX_OBSERVED_TURNS]
    features = {}
    for stat in STATS:
        features[f'p1_mean_{stat}'] = float(np.mean([
            pokemon.get(f'base_{stat}', 0) for pokemon in roster
        ])) if roster else 0.0
    type_counts = Counter(
        str(kind).lower()
        for pokemon in roster
        for kind in pokemon.get('types', [])
        if str(kind).lower() != 'notype'
    )
    unknown_types = set(type_counts) - set(TYPES)
    if unknown_types:
        raise ValueError(f'Unknown real roster types: {sorted(unknown_types)}')
    for kind in TYPES:
        features[f'p1_type_{kind}_count'] = type_counts[kind]
    features['p1_team_diversity'] = len(type_counts)
    for stat in STATS:
        features[f'p2_lead_{stat}'] = lead.get(f'base_{stat}', 0)

    fainted = {'p1': set(), 'p2': set()}
    previous_hp = {'p1': {}, 'p2': {}}
    previous_name = {'p1': None, 'p2': None}
    boost_sums = {'p1': [], 'p2': []}
    for side in ('p1', 'p2'):
        features[f'{side}_no_of_moves'] = 0
        features[f'{side}_dmg'] = 0.0
        features[f'{side}_heal'] = 0.0
        features[f'{side}_switch'] = 0

    for turn in timeline:
        for side in ('p1', 'p2'):
            opponent = 'p2' if side == 'p1' else 'p1'
            state = turn.get(f'{side}_pokemon_state') or {}
            name = state.get('name')
            boosts = state.get('boosts') or {}
            boost_sums[side].append(sum(boosts.get(stat, 0) for stat in BOOST_STATS))
            features[f'{side}_no_of_moves'] += turn.get(f'{side}_move_details') is not None
            if not name:
                previous_name[side] = None
                continue
            if previous_name[side] is not None and previous_name[side] != name:
                features[f'{side}_switch'] += 1
            previous_name[side] = name
            hp = state.get('hp_pct')
            if hp == 0 or state.get('status') == 'fnt':
                fainted[side].add(name)
            if hp is not None:
                hp = float(hp)
                old_hp = previous_hp[side].get(name, 1.0)
                features[f'{opponent}_dmg'] += max(old_hp - hp, 0.0)
                features[f'{side}_heal'] += max(hp - old_hp, 0.0)
                previous_hp[side][name] = hp

    for side in ('p1', 'p2'):
        features[f'{side}_alive_at_turn30'] = TEAM_SIZE - len(fainted[side])
        features[f'{side}_mean_boosts'] = float(np.mean(boost_sums[side])) if boost_sums[side] else 0.0
    features['alive_diff_at_turn30'] = features['p1_alive_at_turn30'] - features['p2_alive_at_turn30']
    features.update(extract_type_effectiveness(battle))
    assert set(features) == set(FEATURE_COLUMNS)
    return features



FEATURE_FUNCTIONS = {"selected": extract_selected_features}
print("Selected numeric inputs:", len(FEATURE_COLUMNS))
display(pd.DataFrame({"feature": FEATURE_COLUMNS}))


### Move-type effectiveness references

Run `python -m pokemon_challenge.preparation.pokemon_details` once before this
notebook to prepare the train-only species lookup. This cell loads that saved
lookup and `DATA_DIR/type_chart_gen2.csv`; it does not re-extract Pokémon details.

For each non-null move in the first 30 entries, compare its **move type** with the
opponent's recorded active Pokémon types. Multiply both defending-type multipliers
for a dual-type Pokémon, ignoring `notype`. The supplied chart is sparse: an
unlisted pair of valid types has multiplier 1.

- Combined multiplier > 1: **+1**.
- Combined multiplier = 1: **0**.
- Combined multiplier < 1 (including immunity): **−1**.

Each side's score is the sum over its non-null moves, including status moves. The unlisted Curse type (`threequestionmarks`, or `curse`)
contributes **0** to these totals. This means its type is ignored, not that its
actual effectiveness is known to be neutral. The move itself is retained in move
counts, boosts, and every other existing feature. No moves are filtered by name. This is a type-matchup proxy, not a damage calculation or a simulation
of move-specific status rules. It uses the recorded opponent state for that turn;
the data does not identify a separate historical target for every move.

Battle-local P1 roster/P2 lead types take precedence; other species use the saved
lookup. If any scored defender has unknown types, that side's entire total is
`NaN`, rather than treating unknowns as neutral or reporting an incomplete sum.
The existing training-fold imputer handles these missing totals. Null moves are
skipped. The ignored Curse type contributes zero even if the defender is unknown,
because no type comparison is performed for it. Empty timelines or only ignored
types/null moves give a total of zero. Other moves with unavailable types still
produce missing totals. Other invalid type names raise an error.


In [ ]:
TYPE_EFFECTIVENESS_COLUMNS = ("p1_type_effectiveness", "p2_type_effectiveness")
POKEMON_DETAILS_PATH = default_output_dir() / "pokemon_details.json"
TYPE_CHART_PATH = DATA_DIR / "type_chart_gen2.csv"
POKEMON_DETAILS = load_pokemon_details(POKEMON_DETAILS_PATH)


def load_type_chart(path):
    chart = pd.read_csv(path)
    if list(chart.columns) != ["attacking_type", "defending_type", "multiplier"]:
        raise ValueError("Unexpected type-chart columns")
    for column in ("attacking_type", "defending_type"):
        if not chart[column].map(lambda value: isinstance(value, str)).all():
            raise ValueError("Type-chart names must be strings")
        chart[column] = chart[column].str.strip().str.lower()
        if not chart[column].isin(TYPES).all():
            raise ValueError("Unknown type in the supplied type chart")
    if chart.duplicated(["attacking_type", "defending_type"]).any():
        raise ValueError("Duplicate attacking/defending pair in the type chart")
    multipliers = pd.to_numeric(chart["multiplier"], errors="raise")
    if not np.isfinite(multipliers).all() or (multipliers < 0).any():
        raise ValueError("Type multipliers must be finite and nonnegative")
    return {(attack, defend): float(multiplier)
            for attack, defend, multiplier in zip(
                chart["attacking_type"], chart["defending_type"], multipliers)}


TYPE_CHART = load_type_chart(TYPE_CHART_PATH)
# Save the exact inputs used by this run, even if files are later regenerated.
REFERENCE_BYTES = {
    "pokemon_details.json": POKEMON_DETAILS_PATH.read_bytes(),
    "type_chart_gen2.csv": TYPE_CHART_PATH.read_bytes(),
}
REFERENCE_HASHES = {name: hashlib.sha256(content).hexdigest()
                    for name, content in REFERENCE_BYTES.items()}


def type_effectiveness_score(move_type, defending_types):
    move_type = str(move_type or "").strip().lower()
    # Ignore the unlisted Curse type only; retain the move in all features.
    if move_type in {"curse", "threequestionmarks"}:
        return 0
    if move_type == "":
        return np.nan
    if move_type not in TYPES:
        raise ValueError(f"Unknown or missing move type: {move_type!r}")
    if not defending_types:
        return np.nan
    real_types = {str(kind).strip().lower() for kind in defending_types} - {"notype"}
    if not real_types:
        return np.nan
    if not real_types.issubset(TYPES) or len(real_types) > 2:
        raise ValueError(f"Invalid defender types: {defending_types!r}")
    multiplier = float(np.prod([TYPE_CHART.get((move_type, kind), 1.0)
                                for kind in real_types]))
    return int(multiplier > 1.0) - int(multiplier < 1.0)


def extract_type_effectiveness(battle):
    initial = list(battle.get("p1_team_details") or [])
    if battle.get("p2_lead_details"):
        initial.append(battle["p2_lead_details"])
    local_details = {str(p.get("name", "")).strip().lower(): p for p in initial}
    totals = {side: 0 for side in ("p1", "p2")}
    unknown = {side: False for side in totals}
    for turn in (battle.get("battle_timeline") or [])[:MAX_OBSERVED_TURNS]:
        for side, opponent in (("p1", "p2"), ("p2", "p1")):
            move = turn.get(f"{side}_move_details")
            if move is None:
                continue
            state = turn.get(f"{opponent}_pokemon_state") or {}
            name = str(state.get("name") or "").strip().lower()
            detail = local_details.get(name) or POKEMON_DETAILS.get(name) or {}
            score = type_effectiveness_score(move.get("type"), detail.get("types"))
            if np.isnan(score):
                unknown[side] = True
            else:
                totals[side] += score
    return {f"{side}_type_effectiveness": np.nan if unknown[side] else totals[side]
            for side in totals}


print("Prepared Pokémon:", len(POKEMON_DETAILS))
print("Non-neutral chart entries:", len(TYPE_CHART))


## Copy Aimilia's and Leonard's active feature definitions

The following cells copy the feature calculations into this notebook. Helper names
are prefixed to avoid overwriting Amir's constants or functions. Their original
notebooks/modules remain unchanged and are not imported at runtime.

- **Amir:** 45 columns, including the two effectiveness totals. Curse's unlisted
  type contributes zero; the move itself remains in all other features.
- **Aimilia:** 106 columns = static 37 + timeline 35 + boosts 13 + roster 21.
  Her notebook keeps only boost columns from the effects module, so its 24 unused
  effect-rate columns are not registered as features here.
- **Leonard:** 32 columns = team stats 13 + timeline summary 19, shared by his
  modelling notebooks.

All **183 source-prefixed columns** are retained, including duplicates and constants,
so the covariance matrix exposes overlap. Feature functions see only permitted
inputs; wrapper IDs are placeholders removed from their output. Calculations use
the first 30 entries. Aimilia's checkpoint features expect the supplied 30-turn
schema, including turns 10, 20, and 30.


In [ ]:
# Copied from notebooks/aimilia/features_static.py; helper names prefixed to avoid collisions.
_aimilia_static_STATS = ['hp', 'atk', 'def', 'spa', 'spd', 'spe']
_aimilia_static_NOT_A_TYPE = 'notype'

def _aimilia_static_real_types(poke):
    """The Pokemon's actual types, with the 'notype' placeholder stripped out."""
    return [t for t in poke['types'] if t != _aimilia_static_NOT_A_TYPE]

def _aimilia_static_effective_stats(poke):
    """
    Base stats scaled by level.

    Base stats are the species' innate values and do NOT include level. 99.3%
    of Pokemon in this data are level 100, but 422 rows are not (abra, pineco,
    elekid, ...). Left unscaled, a level-5 abra looks exactly as strong as a
    level-100 abra, which is wrong.

    level/100 is a crude linear proxy for the real Gen-2 stat formula, which is
    roughly proportional to level. It is an approximation, but it is much closer
    to the truth than ignoring level entirely, and it costs nothing.
    """
    scale = poke['level'] / 100.0
    return {s: poke[f'base_{s}'] * scale for s in _aimilia_static_STATS}

def _aimilia_static_battle_to_row(battle, labelled):
    row = {'battle_id': battle['battle_id']}
    team = battle['p1_team_details']
    lead = battle['p2_lead_details']
    team_stats = [_aimilia_static_effective_stats(p) for p in team]
    for s in _aimilia_static_STATS:
        vals = [ts[s] for ts in team_stats]
        row[f'p1_{s}_mean'] = sum(vals) / len(vals)
        row[f'p1_{s}_max'] = max(vals)
    row['p1_level_mean'] = sum((p['level'] for p in team)) / len(team)
    row['p1_level_min'] = min((p['level'] for p in team))
    row['p1_n_types'] = len({t for p in team for t in _aimilia_static_real_types(p)})
    row['p1_n_dual_type'] = sum((1 for p in team if len(_aimilia_static_real_types(p)) == 2))
    row['p1_n_special_item'] = sum((1 for p in team if p['item'] != 'leftovers'))
    lead_stats = _aimilia_static_effective_stats(lead)
    for s in _aimilia_static_STATS:
        row[f'p2_lead_{s}'] = lead_stats[s]
    row['p2_lead_level'] = lead['level']
    row['p2_lead_n_types'] = len(_aimilia_static_real_types(lead))
    for s in _aimilia_static_STATS:
        row[f'diff_{s}_mean'] = row[f'p1_{s}_mean'] - row[f'p2_lead_{s}']
        row[f'diff_{s}_max'] = row[f'p1_{s}_max'] - row[f'p2_lead_{s}']
    if labelled:
        row['player_won'] = battle['player_won']
    return row

In [ ]:
# Copied from notebooks/aimilia/features_timeline.py; helper names prefixed to avoid collisions.
_aimilia_timeline_ALIVE_STATUSES = {'nostatus'}
_aimilia_timeline_EARLY = 10
_aimilia_timeline_LATE = 20

def _aimilia_timeline_side_features(turns, side):
    """
    Summarise one player's 30 turns. `side` is 'p1' or 'p2'.

    Note: each turn records only the ACTIVE Pokemon's state, not the whole team.
    So hp_pct jumps around as Pokemon switch in and out - it is the health of
    whoever is on the field, not a team health bar.
    """
    f = {}
    states = [t[f'{side}_pokemon_state'] for t in turns]
    moves = [t[f'{side}_move_details'] for t in turns]
    hp = [s['hp_pct'] for s in states]
    names = [s['name'] for s in states]
    f['faints'] = len({s['name'] for s in states if s['status'] == 'fnt'})
    f['hppct_mean'] = sum(hp) / len(hp)
    f['hppct_final'] = hp[-1]
    f['hppct_min'] = min(hp)
    early = [h for h, t in zip(hp, turns) if t['turn'] <= _aimilia_timeline_EARLY]
    late = [h for h, t in zip(hp, turns) if t['turn'] > _aimilia_timeline_LATE]
    f['hppct_early_mean'] = sum(early) / len(early)
    f['hppct_late_mean'] = sum(late) / len(late)
    f['hppct_trend'] = f['hppct_late_mean'] - f['hppct_early_mean']
    ail = sum((1 for s in states if s['status'] not in _aimilia_timeline_ALIVE_STATUSES and s['status'] != 'fnt'))
    f['status_rate'] = ail / len(states)
    f['slp_rate'] = sum((1 for s in states if s['status'] == 'slp')) / len(states)
    f['switches'] = sum((1 for a, b in zip(names, names[1:]) if a != b))
    f['n_species_seen'] = len(set(names))
    f['no_move_rate'] = sum((1 for m in moves if m is None)) / len(moves)
    acted = [m for m in moves if m is not None]
    if acted:
        f['mean_base_power'] = sum((m['base_power'] for m in acted)) / len(acted)
        f['mean_accuracy'] = sum((m['accuracy'] for m in acted)) / len(acted)
        f['status_move_rate'] = sum((1 for m in acted if m['category'] == 'status')) / len(acted)
    else:
        f['mean_base_power'] = 0.0
        f['mean_accuracy'] = 0.0
        f['status_move_rate'] = 0.0
    return f

def _aimilia_timeline_battle_to_row(battle):
    turns = battle['battle_timeline']
    row = {'battle_id': battle['battle_id']}
    for side in ('p1', 'p2'):
        for k, v in _aimilia_timeline_side_features(turns, side).items():
            row[f'{side}_{k}'] = v
    for k in ('hppct_mean', 'hppct_final', 'status_rate', 'no_move_rate', 'switches'):
        row[f'diff_{k}'] = row[f'p1_{k}'] - row[f'p2_{k}']
    return row

In [ ]:
# Copied from notebooks/aimilia/features_effects.py; helper names prefixed to avoid collisions.
_aimilia_effects_TRACKED_EFFECTS = ['substitute', 'trapped', 'confusion', 'leechseed', 'protect', 'perish', 'attract', 'encore', 'destinybond', 'clamp']
_aimilia_effects_BOOST_STATS = ['atk', 'def', 'spa', 'spd', 'spe']

def _aimilia_effects_side_features(turns, side):
    """Summarise one player's effects and boosts across the 30 turns."""
    features = {}
    states = [turn[f'{side}_pokemon_state'] for turn in turns]
    n_turns = len(states)
    for effect in _aimilia_effects_TRACKED_EFFECTS:
        count = 0
        for state in states:
            if effect in state['effects']:
                count += 1
        features[f'eff_{effect}'] = count / n_turns
    any_effect = 0
    for state in states:
        real_effects = [e for e in state['effects'] if e != 'noeffect']
        if real_effects:
            any_effect += 1
    features['eff_any'] = any_effect / n_turns
    totals = []
    offensive = []
    defensive = []
    speed = []
    for state in states:
        boosts = state['boosts']
        totals.append(sum((boosts[s] for s in _aimilia_effects_BOOST_STATS)))
        offensive.append(boosts['atk'] + boosts['spa'])
        defensive.append(boosts['def'] + boosts['spd'])
        speed.append(boosts['spe'])
    features['boost_mean'] = sum(totals) / n_turns
    features['boost_max'] = max(totals)
    features['boost_final'] = totals[-1]
    features['boost_offensive_mean'] = sum(offensive) / n_turns
    features['boost_defensive_mean'] = sum(defensive) / n_turns
    features['boost_speed_mean'] = sum(speed) / n_turns
    return features

def _aimilia_effects_battle_to_row(battle):
    row = {'battle_id': battle['battle_id']}
    turns = battle['battle_timeline']
    for side in ('p1', 'p2'):
        for name, value in _aimilia_effects_side_features(turns, side).items():
            row[f'{side}_{name}'] = value
    for name in ('eff_substitute', 'eff_trapped', 'boost_mean'):
        row[f'diff_{name}'] = row[f'p1_{name}'] - row[f'p2_{name}']
    return row

In [ ]:
# Copied from notebooks/aimilia/features_roster.py; helper names prefixed to avoid collisions.
_aimilia_roster_TEAM_SIZE = 6
_aimilia_roster_CHECKPOINTS = [10, 20, 30]

def _aimilia_roster_track_rosters(battle):
    """
    Replay the 30 turns and record each side's team state at the checkpoints.

    Returns two dictionaries, one per side, of the form
        {turn: (number_alive, total_hp)}
    """
    p1_roster = {p['name']: 1.0 for p in battle['p1_team_details']}
    p2_roster = {battle['p2_lead_details']['name']: 1.0}
    p1_snapshots = {}
    p2_snapshots = {}
    for turn in battle['battle_timeline']:
        s1 = turn['p1_pokemon_state']
        s2 = turn['p2_pokemon_state']
        p1_roster[s1['name']] = s1['hp_pct']
        p2_roster[s2['name']] = s2['hp_pct']
        if turn['turn'] in _aimilia_roster_CHECKPOINTS:
            p1_snapshots[turn['turn']] = _aimilia_roster_summarise(p1_roster)
            p2_snapshots[turn['turn']] = _aimilia_roster_summarise(p2_roster)
    return (p1_snapshots, p2_snapshots)

def _aimilia_roster_summarise(roster):
    """
    Turn a roster into (how many are alive, total health of the team).

    Pokemon we have never seen count as alive and at full health, which is
    what "not yet sent out" means.
    """
    never_seen = max(0, _aimilia_roster_TEAM_SIZE - len(roster))
    alive = never_seen
    total_hp = float(never_seen)
    for hp in roster.values():
        if hp > 0:
            alive += 1
        total_hp += hp
    return (alive, total_hp)

def _aimilia_roster_battle_to_row(battle):
    row = {'battle_id': battle['battle_id']}
    p1_snapshots, p2_snapshots = _aimilia_roster_track_rosters(battle)
    for turn in _aimilia_roster_CHECKPOINTS:
        p1_alive, p1_hp = p1_snapshots[turn]
        p2_alive, p2_hp = p2_snapshots[turn]
        row[f'p1_alive_t{turn}'] = p1_alive
        row[f'p2_alive_t{turn}'] = p2_alive
        row[f'alive_diff_t{turn}'] = p1_alive - p2_alive
        row[f'p1_teamhp_t{turn}'] = p1_hp
        row[f'p2_teamhp_t{turn}'] = p2_hp
        row[f'teamhp_diff_t{turn}'] = p1_hp - p2_hp
    row['p1_teamhp_decline'] = row['p1_teamhp_t10'] - row['p1_teamhp_t30']
    row['p2_teamhp_decline'] = row['p2_teamhp_t10'] - row['p2_teamhp_t30']
    row['teamhp_decline_diff'] = row['p1_teamhp_decline'] - row['p2_teamhp_decline']
    return row

In [ ]:
# Copied from src/pokemon_challenge/features/basic.py; helper names prefixed to avoid collisions.
def _leonard_team_stats(battle):
    """Summarize all six base stats of P1's team and P2's initial lead."""
    team = battle.get('p1_team_details') or []
    lead = battle.get('p2_lead_details') or {}
    features = {'team_size': len(team)}
    for stat in ('hp', 'atk', 'def', 'spa', 'spd', 'spe'):
        values = [p.get(f'base_{stat}', np.nan) for p in team]
        features[f'p1_mean_{stat}'] = float(np.mean(values)) if values else np.nan
        features[f'p2_lead_{stat}'] = lead.get(f'base_{stat}', np.nan)
    return features

def _leonard_timeline_summary(battle):
    """Summarize recorded active states; HP is not total remaining team health."""
    turns = battle.get('battle_timeline') or []
    features = {'observed_turns': len(turns)}
    for side in ('p1', 'p2'):
        states = [t.get(f'{side}_pokemon_state') or {} for t in turns]
        moves = [t.get(f'{side}_move_details') for t in turns]
        hp = [s['hp_pct'] for s in states if s.get('hp_pct') is not None]
        names = [s.get('name') for s in states]
        features[f'{side}_mean_active_hp'] = float(np.mean(hp)) if hp else np.nan
        features[f'{side}_last_active_hp'] = states[-1].get('hp_pct', np.nan) if states else np.nan
        features[f'{side}_distinct_observed_species'] = len({n for n in names if n is not None})
        features[f'{side}_active_name_changes'] = sum((a is not None and b is not None and (a != b) for a, b in zip(names, names[1:])))
        features[f'{side}_null_moves'] = sum((m is None for m in moves))
        features[f'{side}_status_observations'] = sum((s.get('status') not in (None, 'nostatus', '') for s in states))
        for category in ('physical', 'special', 'status'):
            features[f'{side}_{category}_moves'] = sum(((m or {}).get('category') == category for m in moves))
    return features

In [ ]:
def _aimilia_input(battle):
    return {**battle, "battle_id": 0,
            "battle_timeline": (battle.get("battle_timeline") or [])[:MAX_OBSERVED_TURNS]}


def aimilia_static_features(battle):
    row = _aimilia_static_battle_to_row(_aimilia_input(battle), labelled=False)
    return {key: value for key, value in row.items() if key != "battle_id"}


def aimilia_timeline_features(battle):
    row = _aimilia_timeline_battle_to_row(_aimilia_input(battle))
    return {key: value for key, value in row.items() if key != "battle_id"}


def aimilia_boost_features(battle):
    row = _aimilia_effects_battle_to_row(_aimilia_input(battle))
    return {key: value for key, value in row.items() if "boost" in key}


def aimilia_roster_features(battle):
    row = _aimilia_roster_battle_to_row(_aimilia_input(battle))
    return {key: value for key, value in row.items() if key != "battle_id"}


def leonard_team_features(battle):
    return _leonard_team_stats(battle)


def leonard_timeline_features(battle):
    observed = {**battle, "battle_timeline":
                (battle.get("battle_timeline") or [])[:MAX_OBSERVED_TURNS]}
    return _leonard_timeline_summary(observed)


FEATURE_FUNCTIONS.update({
    "aimilia_static": aimilia_static_features,
    "aimilia_timeline": aimilia_timeline_features,
    "aimilia_boosts": aimilia_boost_features,
    "aimilia_roster": aimilia_roster_features,
    "leonard_team": leonard_team_features,
    "leonard_timeline": leonard_timeline_features,
})
EXPECTED_GROUP_SIZES = {"selected": 45, "aimilia_static": 37, "aimilia_timeline": 35,
                        "aimilia_boosts": 13, "aimilia_roster": 21,
                        "leonard_team": 13, "leonard_timeline": 19}
ALL_FEATURE_GROUPS = tuple(FEATURE_FUNCTIONS)

COPIED_FEATURE_SOURCE_SHA256 = {'notebooks/aimilia/features_static.py': '932e3f2866af8824e708a65b740650be7b3bcc9d29a478ce0e59bd4033d914eb', 'notebooks/aimilia/features_timeline.py': '73661ad55b5461bd8ea6ce08f051c24f528e238fbf4d8ecfc8ca1cd373ad0a0b', 'notebooks/aimilia/features_effects.py': 'be72c0ba03ace0097675379d7e427fa32ecafd27c02e2d789211ffa3cde3744b', 'notebooks/aimilia/features_roster.py': '31448cd844852f74a2ecb420d6107412af5dc8c2e16b59c26c62011e042855b8', 'src/pokemon_challenge/features/basic.py': '1caa86ddd1aa464a0ea96c01a26a795a8fb57227392441e7cf87b2abccf4f00f'}
display(pd.Series(EXPECTED_GROUP_SIZES, name="feature columns").to_frame())


## 2. Load features and use the shared validation settings

Inputs are located by the project's shared helper: local `data/raw/` or `data/`,
the supported Kaggle paths, or an explicit `POKEMON_DATA_DIR`. Dataset location
never determines the output directory. To reuse a dataset stored elsewhere, set
`POKEMON_DATA_DIR` to that folder before starting the notebook kernel.

The unchanged `configs/validation.json` defaults reserve 20% of labelled battles
as a holdout and use five stratified folds within the other 80%, with seed 42.
Every candidate uses the same saved assignments. Imputation, constant removal,
and Logistic Regression scaling are fitted using each fold's training rows only.

**Validation change from the old notebook:** this version uses all training rows
and the existing shared splitter. It does not exclude training/test content matches
or group duplicate inputs. Duplicate-policy improvements are deferred; the shared
validation code is unchanged. Old grouped-CV and holdout scores are not comparable
to results from this setup. Test data is read only when generating a submission.

Rerun setup to obtain a new run name before constructing a new experiment.


## Remove redundant features before modelling

The notebook still extracts the 183 raw features so every fitted model receives
the same input schema. Inside each model pipeline, a training-only filter removes:

1. Constant or entirely missing columns.
2. Exact duplicate columns (including matching missing-value positions).
3. Later columns with **absolute Pearson correlation ≥ 0.95** against a retained
   column. This includes strong negative relationships.

Correlation is scale-independent; large raw covariance alone does not establish
redundancy. Change `CORRELATION_THRESHOLD` here if needed. The first feature wins
in input order: Amir first, then Aimilia, then Leonard; keys within groups are
alphabetical. No labels enter these decisions.

Each CV fold learns its own removals using only its training rows. The holdout fit
learns them from development rows, and a final submission fit learns them from all
labelled rows. A development-only preview below explains removals and draws the
reduced matrices; it does not preselect inputs for CV. Retain this class-definition
cell when reloading `model.joblib`, since the saved pipeline uses the notebook class.


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.utils.validation import check_is_fitted

CORRELATION_THRESHOLD = 0.95


class RedundancyFilter(TransformerMixin, BaseEstimator):
    """Keep first nonconstant, nonduplicate, weakly correlated numeric columns."""

    def __init__(self, correlation_threshold=0.95):
        self.correlation_threshold = correlation_threshold

    def fit(self, X, y=None):
        if not 0 < self.correlation_threshold <= 1:
            raise ValueError("correlation_threshold must be in (0, 1]")
        if not isinstance(X, pd.DataFrame) or X.columns.has_duplicates:
            raise ValueError("Pass a DataFrame with distinct feature names")
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.n_features_in_ = X.shape[1]
        decisions = {}
        unique = []
        for name in X:
            if X[name].nunique(dropna=True) <= 1:
                decisions[name] = {"action": "constant", "reference_feature": None,
                                   "correlation": np.nan}
                continue
            duplicate = next((prior for prior in unique if X[name].equals(X[prior])), None)
            if duplicate is not None:
                decisions[name] = {"action": "duplicate", "reference_feature": duplicate,
                                   "correlation": 1.0}
            else:
                unique.append(name)
        correlations = X[unique].corr(method="pearson", min_periods=2)
        kept = []
        for name in unique:
            reference = next((prior for prior in kept
                              if pd.notna(correlations.at[name, prior])
                              and abs(correlations.at[name, prior]) >= self.correlation_threshold), None)
            if reference is None:
                kept.append(name)
                decisions[name] = {"action": "kept", "reference_feature": name,
                                   "correlation": np.nan}
            else:
                decisions[name] = {"action": "high_correlation", "reference_feature": reference,
                                   "correlation": float(correlations.at[name, reference])}
        if not kept:
            raise ValueError("No nonconstant features remain after redundancy filtering")
        self.kept_features_ = np.asarray(kept, dtype=object)
        self.support_ = np.isin(self.feature_names_in_, self.kept_features_)
        self.removal_report_ = pd.DataFrame([
            {"feature": name, **decisions[name]} for name in self.feature_names_in_])
        return self

    def transform(self, X):
        check_is_fitted(self, "kept_features_")
        if not isinstance(X, pd.DataFrame) or list(X.columns) != list(self.feature_names_in_):
            raise ValueError("Prediction columns must match the fitted input names and order")
        return X.loc[:, self.kept_features_].copy()

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, "kept_features_")
        if input_features is not None and not np.array_equal(input_features, self.feature_names_in_):
            raise ValueError("input_features do not match the fitted feature names")
        return self.kept_features_.copy()

    def get_support(self, indices=False):
        check_is_fitted(self, "support_")
        return np.flatnonzero(self.support_) if indices else self.support_.copy()


class AmirFilteredExperiment(BattleExperiment):
    """Notebook-local extension; teammates' runner and pipelines stay unchanged."""

    def __init__(self, *args, correlation_threshold=0.95, **kwargs):
        self.correlation_threshold = correlation_threshold
        super().__init__(*args, **kwargs)
        self.manifest["feature_filter"] = {
            "correlation_threshold": self.correlation_threshold,
            "order": ["constants", "exact_duplicates", "absolute_pearson_correlation"],
            "priority": "first input column; Amir, Aimilia, Leonard; sorted keys within groups",
            "fit_scope": "each CV training fold; development for holdout; all labels for final fit",
        }
        text = json.dumps(self.manifest, indent=2) + "\n"
        (self.output_path / "experiment.json").write_text(text, encoding="utf-8")
        (self.output_path / "preprocessing.json").write_text(
            json.dumps(self.manifest["feature_filter"], indent=2) + "\n", encoding="utf-8")

    def _pipeline(self, candidate):
        base = super()._pipeline(candidate)
        return Pipeline([
            ("redundancy_filter", RedundancyFilter(self.correlation_threshold)),
            *base.steps,
        ])


In [ ]:
experiment = AmirFilteredExperiment(
    owner=OWNER,
    run=RUN,
    feature_functions=FEATURE_FUNCTIONS,
    data_path=DATA_DIR,
    correlation_threshold=CORRELATION_THRESHOLD,
)
assert experiment.X.shape[1] == sum(EXPECTED_GROUP_SIZES.values()) == 183
assert {name: sum(column.startswith(name + "__") for column in experiment.X)
        for name in FEATURE_FUNCTIONS} == EXPECTED_GROUP_SIZES
assert {column for column in experiment.X if column.startswith("selected__")} == {
    f"selected__{name}" for name in FEATURE_COLUMNS}
original_columns = [f"selected__{name}" for name in FEATURE_COLUMNS
                    if name not in TYPE_EFFECTIVENESS_COLUMNS]
if not np.isfinite(experiment.X[original_columns].to_numpy()).all():
    raise ValueError("The original 43 training features must be finite")
effectiveness_columns = [f"selected__{name}" for name in TYPE_EFFECTIVENESS_COLUMNS]
if np.isinf(experiment.X[effectiveness_columns].to_numpy()).any():
    raise ValueError("Infinite type-effectiveness total")
display(experiment.X[effectiveness_columns].isna()
        .sum().rename("battles with unscorable type matchups").to_frame())
for filename, content in REFERENCE_BYTES.items():
    (experiment.output_path / filename).write_bytes(content)
(experiment.output_path / "reference_inputs.json").write_text(
    json.dumps({"sha256": REFERENCE_HASHES,
                "unknown_policy": "entire side total is NaN for unknown defender or move type; fold-fitted median imputation",
                "score": "+1 if multiplier>1; 0 if multiplier==1; -1 if multiplier<1",
                "includes_status_moves": True,
                "ignored_move_types": ["curse", "threequestionmarks"],
                "ignored_type_contribution": 0,
                "moves_filtered_by_name": False}, indent=2) + "\n", encoding="utf-8"
)

display(experiment.X.head())
print("Numeric input features:", experiment.X.shape[1])
display(experiment.assignments.groupby(["partition", "validation_fold"])
        .size().rename("battles").to_frame())

(experiment.output_path / "copied_feature_sources.json").write_text(
    json.dumps(COPIED_FEATURE_SOURCE_SHA256, indent=2) + "\n", encoding="utf-8")


## Feature covariance matrix — development battles only

The **raw sample covariance** is calculated from the 8,000 development battles,
before imputation/scaling, with `ddof=1` and pairwise deletion of missing values.
Each entry is in the product of its two features' original units. The diagonal is
the sample variance. A pairwise observation-count matrix is saved alongside it.
Pairwise deletion can produce a matrix that is not positive semidefinite; this
diagnostic is not used as an inverse/whitening transform.

All 183 features remain in source order. The covariance overview uses a labelled
symmetric-log **colour scale only** to make small and large raw covariances visible;
the CSV contains untransformed numbers. Group boundaries identify feature blocks.
The full labelled SVG can be zoomed, and `plot_covariance_block(row_group,
column_group)` below gives a labelled source-block view.

A separate **Pearson correlation** view helps compare relationships across scales.
Constants have zero covariance and undefined (masked) correlation. Exact duplicate
columns and highly correlated pairs are listed for inspection; nothing is removed
or selected using this matrix. No holdout or official test rows enter this analysis.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm, TwoSlopeNorm

X_development = experiment.X.iloc[experiment.development].copy()
assert not {"battle_id", "player_won"}.intersection(X_development.columns)
feature_covariance = X_development.cov(ddof=1, min_periods=2)
feature_correlation = X_development.corr(method="pearson", min_periods=2)
observed = X_development.notna().astype("int64")
pairwise_observations = observed.T.dot(observed)
assert feature_covariance.shape == (183, 183)
assert feature_covariance.index.equals(X_development.columns)
assert np.allclose(feature_covariance, feature_covariance.T, equal_nan=True)
assert np.allclose(np.diag(feature_covariance), X_development.var(ddof=1), equal_nan=True)
assert np.isfinite(feature_covariance.to_numpy()).all()

feature_catalog = pd.DataFrame({
    "position": np.arange(1, X_development.shape[1] + 1),
    "column": list(X_development),
    "group": [column.split("__", 1)[0] for column in X_development],
    "feature": [column.split("__", 1)[1] for column in X_development],
    "nonmissing_rows": X_development.notna().sum().to_numpy(),
    "sample_variance": X_development.var(ddof=1).to_numpy(),
})
constant_features = feature_catalog.loc[feature_catalog.sample_variance.eq(0), "column"].tolist()
duplicate_rows = []
columns = list(X_development)
for i, left in enumerate(columns):
    for right in columns[i+1:]:
        if X_development[left].equals(X_development[right]):
            duplicate_rows.append({"feature_a": left, "feature_b": right})
exact_duplicates = pd.DataFrame(duplicate_rows, columns=["feature_a", "feature_b"])
correlation_pairs = feature_correlation.where(
    np.triu(np.ones(feature_correlation.shape, dtype=bool), k=1)
).stack().rename("correlation").reset_index()
correlation_pairs.columns = ["feature_a", "feature_b", "correlation"]
correlation_pairs["absolute_correlation"] = correlation_pairs.correlation.abs()
high_correlations = correlation_pairs.loc[correlation_pairs.absolute_correlation.ge(0.95)].sort_values(
    "absolute_correlation", ascending=False)

feature_covariance.to_csv(experiment.output_path / "feature_covariance.csv")
feature_correlation.to_csv(experiment.output_path / "feature_correlation.csv")
pairwise_observations.to_csv(experiment.output_path / "covariance_pairwise_counts.csv")
feature_catalog.to_csv(experiment.output_path / "feature_catalog.csv", index=False)
exact_duplicates.to_csv(experiment.output_path / "exact_duplicate_features.csv", index=False)
high_correlations.to_csv(experiment.output_path / "high_correlation_pairs.csv", index=False)
summary = {
    "development_rows": len(X_development), "features": len(columns),
    "group_sizes": EXPECTED_GROUP_SIZES,
    "covariance_ddof": 1, "missing_policy": "pairwise complete observations",
    "minimum_pairwise_rows": int(pairwise_observations.min().min()),
    "maximum_pairwise_rows": int(pairwise_observations.max().max()),
    "constant_features": constant_features, "exact_duplicate_pairs": len(exact_duplicates),
    "pairs_with_absolute_correlation_at_least_0_95": len(high_correlations),
    "train_sha256": experiment.manifest["train_sha256"],
    "splits_sha256": experiment.manifest["splits_sha256"],
}
(experiment.output_path / "covariance_summary.json").write_text(
    json.dumps(summary, indent=2) + "\n", encoding="utf-8")
print(f"{len(X_development):,} development battles; {len(columns)} × {len(columns)} covariance matrix")
print("Constant features:", constant_features)
print("Exact duplicate pairs:", len(exact_duplicates))
display(exact_duplicates.head(15))
display(high_correlations.head(15))

group_labels = {"selected": "Amir (45)", "aimilia_static": "Aimilia static (37)",
                "aimilia_timeline": "Aimilia timeline (35)", "aimilia_boosts": "Aimilia boosts (13)",
                "aimilia_roster": "Aimilia roster (21)", "leonard_team": "Leonard team (13)",
                "leonard_timeline": "Leonard timeline (19)"}
group_sizes = np.array(list(EXPECTED_GROUP_SIZES.values()))
boundaries = np.cumsum(group_sizes) - 0.5
centres = np.cumsum(group_sizes) - group_sizes / 2 - 0.5
cov_limit = float(np.nanmax(np.abs(feature_covariance.to_numpy())))
nonzero = np.abs(feature_covariance.to_numpy())
nonzero = nonzero[nonzero > 0]
cov_linear_threshold = max(float(np.quantile(nonzero, 0.1)), np.finfo(float).eps)
cov_norm = SymLogNorm(linthresh=cov_linear_threshold, vmin=-cov_limit, vmax=cov_limit, base=10)
matrix_cmap = plt.get_cmap("PuOr_r").copy()
matrix_cmap.set_bad("#d9d9d9")


def plot_feature_matrix(matrix, *, correlation=False, full_labels=False):
    size = (48, 44) if full_labels else (11.5, 9.5)
    fig, ax = plt.subplots(figsize=size)
    norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1) if correlation else cov_norm
    image = ax.imshow(np.ma.masked_invalid(matrix.to_numpy()), cmap=matrix_cmap,
                      norm=norm, interpolation="nearest", aspect="equal")
    local_groups = list(dict.fromkeys(name.split("__", 1)[0] for name in matrix.columns))
    local_sizes = np.array([sum(name.startswith(group + "__") for name in matrix.columns)
                            for group in local_groups])
    local_boundaries = np.cumsum(local_sizes) - 0.5
    local_centres = np.cumsum(local_sizes) - local_sizes / 2 - 0.5
    local_labels = [group_labels[group].rsplit(" (", 1)[0] + f" ({size})"
                    for group, size in zip(local_groups, local_sizes)]
    if full_labels:
        ticks = np.arange(len(matrix))
        ax.set_xticks(ticks, matrix.columns, rotation=90, fontsize=9)
        ax.set_yticks(ticks, matrix.index, fontsize=9)
    else:
        ax.set_xticks(local_centres, local_labels, rotation=40, ha="right", fontsize=10)
        ax.set_yticks(local_centres, local_labels, fontsize=10)
    for boundary in local_boundaries[:-1]:
        ax.axhline(boundary, color="#555555", linewidth=0.6)
        ax.axvline(boundary, color="#555555", linewidth=0.6)
    ax.set_xlabel("Feature columns (ordered by source)")
    ax.set_ylabel("Feature columns (ordered by source)")
    kind = "Pearson correlation" if correlation else "Raw sample covariance"
    ax.set_title(f"{kind} · {len(matrix.columns)} features · {len(X_development):,} development battles", pad=16)
    colorbar = fig.colorbar(image, ax=ax, fraction=0.04, pad=0.035)
    colorbar.set_label("Correlation (−1 to +1); grey = undefined" if correlation else
                       "Covariance (original feature-unit products; symmetric-log colour)")
    fig.tight_layout()
    return fig


covariance_figure = plot_feature_matrix(feature_covariance)
covariance_figure.savefig(experiment.output_path / "feature_covariance.png", dpi=180, bbox_inches="tight")
plt.show()
correlation_figure = plot_feature_matrix(feature_correlation, correlation=True)
correlation_figure.savefig(experiment.output_path / "feature_correlation.png", dpi=180, bbox_inches="tight")
plt.show()
with plt.rc_context({"svg.fonttype": "none"}):
    labelled_figure = plot_feature_matrix(feature_covariance, full_labels=True)
    labelled_figure.savefig(experiment.output_path / "feature_covariance_full_labels.svg", bbox_inches="tight")
    plt.close(labelled_figure)


### Inspect a labelled covariance block

Choose row and column source groups, or call `plot_covariance_block` directly.
The full CSV and zoomable labelled SVG remain available in this run's output folder.


In [ ]:
def plot_covariance_block(row_group="selected", column_group="selected"):
    row_names = [column for column in feature_covariance if column.startswith(row_group + "__")]
    column_names = [column for column in feature_covariance if column.startswith(column_group + "__")]
    if not row_names or not column_names:
        raise ValueError("Choose group names from FEATURE_FUNCTIONS")
    block = feature_covariance.loc[row_names, column_names]
    fig, ax = plt.subplots(figsize=(max(10, len(column_names)*0.3), max(8, len(row_names)*0.26)))
    image = ax.imshow(block.to_numpy(), cmap=matrix_cmap, norm=cov_norm, aspect="auto")
    ax.set_xticks(np.arange(len(column_names)), [n.split("__",1)[1] for n in column_names],
                  rotation=90, fontsize=10)
    ax.set_yticks(np.arange(len(row_names)), [n.split("__",1)[1] for n in row_names], fontsize=10)
    ax.set_xlabel(group_labels[column_group])
    ax.set_ylabel(group_labels[row_group])
    ax.set_title("Sample covariance · development battles only")
    fig.colorbar(image, ax=ax, label="Covariance (original units; symmetric-log colour)")
    fig.tight_layout()
    plt.show()
    return block


try:
    import ipywidgets as widgets
except ImportError:
    print("Optional ipywidgets is unavailable. Call plot_covariance_block(row_group, column_group).")
else:
    def _show_covariance_block(row_group, column_group):
        plot_covariance_block(row_group, column_group)
    display(widgets.interactive(_show_covariance_block,
                               row_group=list(FEATURE_FUNCTIONS), column_group=list(FEATURE_FUNCTIONS)))


## Reduced feature set and matrices — preview

This preview fits the same filter on development rows only. It saves every keep/drop
decision, the retained column list, and reduced covariance/correlation matrices.
These columns are used only for this preview; the model comparisons below fit
their own filters independently inside the folds. Counts can differ between folds.
The original full-matrix files remain available for comparison.


In [ ]:
development_filter = RedundancyFilter(CORRELATION_THRESHOLD).fit(X_development)
X_development_filtered = development_filter.transform(X_development)
filter_report = development_filter.removal_report_
kept_features = list(development_filter.get_feature_names_out())
removed_features = filter_report.loc[filter_report.action.ne("kept")].copy()
removal_counts = {key: int(filter_report.action.eq(key).sum())
                  for key in ["constant", "duplicate", "high_correlation", "kept"]}
assert len(kept_features) + len(removed_features) == 183
remaining_correlations = X_development_filtered.corr().abs().to_numpy(copy=True)
np.fill_diagonal(remaining_correlations, np.nan)
assert np.nanmax(remaining_correlations) < CORRELATION_THRESHOLD
filtered_covariance = X_development_filtered.cov(ddof=1, min_periods=2)
filtered_correlation = X_development_filtered.corr(min_periods=2)
assert np.allclose(filtered_covariance, filtered_covariance.T, equal_nan=True)
assert np.allclose(np.diag(filtered_covariance), X_development_filtered.var(ddof=1))

filter_report.to_csv(experiment.output_path / "development_feature_filter.csv", index=False)
removed_features.to_csv(experiment.output_path / "removed_features.csv", index=False)
filtered_covariance.to_csv(experiment.output_path / "filtered_feature_covariance.csv")
filtered_correlation.to_csv(experiment.output_path / "filtered_feature_correlation.csv")
filter_summary = {"scope": "development-only preview, not a CV-wide feature selection",
                  "input_features": 183, "retained_features": len(kept_features),
                  "correlation_threshold": CORRELATION_THRESHOLD,
                  "counts": removal_counts, "kept_features": kept_features}
(experiment.output_path / "feature_filter_summary.json").write_text(
    json.dumps(filter_summary, indent=2) + "\n", encoding="utf-8")
print(f"Development preview: 183 raw columns → {len(kept_features)} retained")
display(pd.Series(removal_counts, name="columns").to_frame())
display(removed_features)

filtered_covariance_figure = plot_feature_matrix(filtered_covariance)
filtered_covariance_figure.savefig(experiment.output_path / "filtered_feature_covariance.png",
                                   dpi=180, bbox_inches="tight")
plt.show()
filtered_correlation_figure = plot_feature_matrix(filtered_correlation, correlation=True)
filtered_correlation_figure.savefig(experiment.output_path / "filtered_feature_correlation.png",
                                    dpi=180, bbox_inches="tight")
plt.show()
with plt.rc_context({"svg.fonttype": "none"}):
    labelled_filtered_figure = plot_feature_matrix(filtered_covariance, full_labels=True)
    labelled_filtered_figure.savefig(experiment.output_path / "filtered_covariance_full_labels.svg",
                                     bbox_inches="tight")
    plt.close(labelled_filtered_figure)


## 3. Compare the original two model configurations

Both candidates accept all seven raw feature groups (183 input columns), remove redundancy inside each training fold, and retain the standalone
notebook's classifier parameters. The runner adds its standard fold-fitted
preprocessing. `percentile=100` disables ANOVA selection. The redundancy filter is enabled
before imputation, constant filtering, scaling, and model fitting.

**CV (cross-validation):** train on four development folds and score on the fifth,
repeating five times so every development battle is evaluated once. Compare the
mean accuracy, each fold's score, and the majority-class reference. The holdout
is excluded from model selection.


In [ ]:
candidates = [
    Candidate(
        "combined_logistic",
        groups=ALL_FEATURE_GROUPS,
        model="logistic_regression",
        percentile=100,
        params={"C": 1.0, "max_iter": 2000},
    ),
    Candidate(
        "combined_hist_boosting",
        groups=ALL_FEATURE_GROUPS,
        model="hist_gradient_boosting",
        percentile=100,
        params={
            "max_iter": 250,
            "learning_rate": 0.06,
            "max_leaf_nodes": 15,
            "min_samples_leaf": 25,
            "l2_regularization": 5.0,
            "early_stopping": False,
        },
    ),
]
results = experiment.compare(candidates)
display(results)
print("Highest mean CV accuracy:", experiment.best_candidate)


## 4. Inspect retained features and fold variation

`cv_accuracy` is the mean of the five validation accuracies. `fold_std` describes
variation between folds; it is not a confidence interval. `dummy_accuracy` is the
majority-class reference on those same folds.

`selected_in_folds` reports how often each input survived the fold-fitted
redundancy filter and the remaining preprocessing. A constant
input can be removed even with `percentile=100`. Add any new uniquely named
candidates through `experiment.compare(...)` before opening the holdout.


In [ ]:
for candidate in candidates:
    path = experiment.output_path / f"selection_frequency_{candidate.name}.csv"
    selection = pd.read_csv(path).sort_values(
        ["selected_in_folds", "feature"], ascending=[False, True]
    )
    print(candidate.name)
    display(selection)


## 5. Final holdout evaluation — explicit decision

Leave this **False** while choosing features and models. Set it to **True** only when you have chosen your approach from CV. The chosen pipeline is fitted on the 80% development subset and evaluated on the untouched 20% holdout.

Opening the holdout locks further comparisons in this experiment. Do not tune on its score or repeatedly create new runs to work around the lock; doing so would turn the holdout into another tuning set. The guard is per Python experiment object, not a cross-team access control system.

In [ ]:
EVALUATE_HOLDOUT = False
CHOSEN_CANDIDATE = experiment.best_candidate  # or a specific compared candidate name

if EVALUATE_HOLDOUT:
    holdout = experiment.evaluate_holdout(CHOSEN_CANDIDATE)
    print("Candidate:", holdout["candidate"])
    print("Holdout accuracy:", holdout["accuracy"])
    display(pd.DataFrame(holdout["confusion_matrix"],
                         index=["actual loss", "actual win"], columns=["predicted loss", "predicted win"]))
    display(pd.DataFrame(holdout["classification_report"]).T)
    print("Selected features:", holdout["selected_features"])
else:
    print("Holdout remains unopened. Continue comparing using CV only.")


## 6. Train on all labels and generate a submission (optional)

After evaluating the chosen candidate, set `MAKE_SUBMISSION=True`. The shared
runner refits the locked configuration on all labelled training battles, including
the former holdout, and predicts `test.jsonl`. Its labels are unknown locally.

The sample submission supplies the IDs and row order. Files are saved as
`submission.csv`, `model.joblib`, and `final_fit.json` inside this run's output
folder; nothing is uploaded automatically. The saved pipeline accepts the recorded
numeric input columns, so retain this notebook's feature functions for raw battles.


In [ ]:
MAKE_SUBMISSION = False
if MAKE_SUBMISSION:
    submission = experiment.make_submission()
    display(submission.head())
    print("Upload manually:", experiment.output_path / "submission.csv")
else:
    print("Submission generation skipped.")


## 7. Artifacts and experiment notes

The unique ignored run folder contains:

- `experiment.json`: validation settings, data/split hashes, versions, code commit and dirty-tree status.
- `feature_sources.json`: feature function source where available.
- `splits.csv`: saved holdout and development-fold assignments.
- `candidates.json`, `cv_results.csv`, and `selection_frequency_*.csv`: configurations and CV evidence.
- After holdout evaluation: metrics, selected features, and battle-level predictions.
- After final fitting: `model.joblib`, `submission.csv`, and `final_fit.json`.

Record the hypothesis, interpretation, and exact run/commit in your notes. Uncommitted changes are flagged in the manifest; commit the implementation and retain this notebook/configuration before relying on a final result. Saved source snippets help auditing but do not automatically package custom imports or globals.

Only load model files you trust. Saved outputs are local and ignored by Git; commit the notebook/shared code separately and preserve final artifacts with your team.

The current run uses the project validation configuration and all labelled rows.
Keep results from the old standalone grouped/excluded-data workflow separate.


The run also saves copies of `pokemon_details.json` and `type_chart_gen2.csv`,
plus their SHA256 hashes and scoring policy in `reference_inputs.json`.

The run also contains feature catalogs, raw covariance/correlation CSVs, pairwise
counts, exact-duplicate diagnostics, and matrix figures from development data only.

Filtering adds `preprocessing.json`, development keep/drop reports, retained feature
lists, and reduced matrix CSVs/figures. Selection-frequency CSVs report the actual
fold-fitted retained columns. Saved pipelines expect all 183 raw input columns and
apply their fitted filter internally.


In [ ]:
print("Run folder:", experiment.output_path)
for path in sorted(experiment.output_path.iterdir()):
    print(" -", path.name)
